Title: BC_parallel_LS.ipynb

Purpose: Bias correct the climate data

Author: Onno Nennecke on 05.11.2026 Modified: 

Input data: 

- ERA5 Data    - These files lie here: '/climca/people/ppfleiderer/ERA5/RL_climate/ERA5_raw/*'
- CMIP6 Data - These files lie here: '/climca/people/onennecke/not_debiased_data' and '/climca/people/onennecke/not_debiased_data_future_2005_2024_ref'
- CMIP6 runs are defined in the csv file: /home/onennecke/CMIP_models/CMIP6_runs.csv
- Data that defines the reference runs '/home/onennecke/CMIP_models/CMIP6_runs_historical.csv'
- Historical CMIP6 and ERA5 data: '/climca/people/onennecke/not_debiased_1970_2000' '/climca/people/onennecke/not_debiased_1970_2000/ERA5_1970_2000'

Output data:

- This file lies here: /climca/people/onennecke/debiased_data/

In [1]:
# Importing libraries
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import cftime
import time
import re
from ibicus.debias import QuantileMapping
from ibicus.debias import ISIMIP
from ibicus.debias import LinearScaling
import multiprocessing
from pathlib import Path


import os
import time
import matplotlib.pyplot as plt
from ibicus.evaluate import assumptions, correlation, marginal, multivariate, trend
from ibicus.evaluate.metrics import *
# Importing functions
import Functions.grid_func as grid_func
import Functions.wind_model_func as wind_model_func

/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [2]:
df = pd.read_csv('/home/onennecke/CMIP_models/CMIP6_runs_historical.csv')
df
# Change run for BCC and CESM2
# df.loc[df['ESM'] == 'BCC-CSM2-MR', 'run'] = 'r1i1p1f1'
# df.loc[df['ESM'] == 'CESM2', 'run'] = 'r10i1p1f1'
df.loc[df['ESM'] == 'CESM2', 'variables'] = 'tas, tasmax, rsds, sfcWind, psl'

df

,ESM,Institution,run,variables
0,ACCESS-CM2,CSIRO-ARCCSS,r1i1p1f1,"tas, tasmax, rsds, sfcWind, psl"
1,BCC-CSM2-MR,BCC,r2i1p1f1,"tas, tasmax, sfcWind, psl"
2,CESM2,NCAR,r1i1p1f1,"tas, tasmax, rsds, sfcWind, psl"
3,EC-Earth3,EC-Earth-Consortium,r1i1p1f1,"tas, tasmax, rsds, sfcWind, psl"
4,GFDL-ESM4,NOAA-GFDL,r1i1p1f1,"tas, tasmax, rsds, sfcWind, psl"
5,MPI-ESM1-2-HR,MPI-M,r1i1p1f1,"tas, tasmax, rsds, sfcWind, psl"
6,MRI-ESM2-0,MRI,r1i1p1f1,"tas, tasmax, rsds, sfcWind, psl"
7,KACE-1-0-G,NIMS-KMA,r1i1p1f1,"tas, tasmax, rsds, sfcWind, psl"
8,TaiESM1,AS-RCEC,r1i1p1f1,"tas, tasmax, rsds, sfcWind, psl"
9,UKESM1-0-LL,MOHC,r1i1p1f2,"tas, tasmax, rsds, sfcWind, psl"


In [3]:
# Read CMIP6 data from 1970-2000
base_path = '/climca/people/onennecke/not_debiased_1970_2000'

all_files = []
for var in ['tas', 'rsds', 'sfcWind', 'tasmax']:
    run = glob.glob(os.path.join(base_path, f"*_{var}.nc"))
    all_files.extend(run)

def add_esm(ds):
    fname = Path(ds.encoding["source"]).name
    esm, run = fname.split("_")[:2]
    ds = ds.expand_dims(ESM=[esm])
    return ds#.assign_coords(
    #     ESM_run=("ESM", f"{esm}_{run}"),
    #     run=("ESM", [run]),
    # )
combined_ds_CMIP6_1970_2000 = xr.open_mfdataset(
    all_files,
    preprocess=add_esm,
    combine="by_coords",
    join="outer"
)

psl_files = glob.glob(os.path.join(base_path, "*_psl.nc"))
psl_ds_CMIP6_1970_2000 = xr.open_mfdataset(
    psl_files,
    preprocess=add_esm,
    combine="by_coords",
    join="outer"
)

In [4]:
# Read ERA5 data from 1970-2000
base_path = '/climca/people/onennecke/not_debiased_1970_2000/ERA5_1970_2000'

all_files = []
for var in ['tas', 'rsds', 'sfcWind', 'tasmax']:
    run = glob.glob(os.path.join(base_path, f"*_{var}.nc"))
    all_files.extend(run)

# def add_esm(ds):
#     fname = Path(ds.encoding["source"]).name
#     esm, run = fname.split("_")[:2]
#     ds = ds.expand_dims(ESM_run=[f"{esm}_{run}"])
#     return ds.assign_coords(
#         ESM=("ESM_run", [esm]),
#         run=("ESM_run", [run]),
#     )
combined_ds_ERA5_1970_2000 = xr.open_mfdataset(
    all_files,
    preprocess=add_esm,
    combine="by_coords",
    join="outer"
)

psl_files = glob.glob(os.path.join(base_path, "*_psl.nc"))
psl_ds_ERA5_1970_2000 = xr.open_mfdataset(
    psl_files,
    preprocess=add_esm,
    combine="by_coords",
    join="outer"
)

In [5]:
combined_ds_ERA5_1970_2000

<xarray.Dataset> Size: 16MB
Dimensions:   (ESM: 1, time: 11315, lat: 9, lon: 10)
Coordinates:
  * ESM       (ESM) object 8B 'ERA5'
  * time      (time) datetime64[ns] 91kB 1970-01-01 1970-01-02 ... 2000-12-31
  * lat       (lat) int64 72B 47 48 49 50 51 52 53 54 55
    crs       int64 8B 4326
    gridtype  <U6 24B 'lonlat'
  * lon       (lon) int64 80B 6 7 8 9 10 11 12 13 14 15
    run       <U4 16B 'hist'
    ESM_run   <U9 36B 'ERA5_hist'
Data variables:
    tas       (ESM, time, lat, lon) float32 4MB dask.array<chunksize=(1, 11315, 9, 10), meta=np.ndarray>
    rsds      (ESM, time, lat, lon) float32 4MB dask.array<chunksize=(1, 11315, 9, 10), meta=np.ndarray>
    sfcWind   (ESM, time, lat, lon) float32 4MB dask.array<chunksize=(1, 11315, 9, 10), meta=np.ndarray>
    tasmax    (ESM, time, lat, lon) float32 4MB dask.array<chunksize=(1, 11315, 9, 10), meta=np.ndarray>

In [6]:
combined_ds_CMIP6_1970_2000

<xarray.Dataset> Size: 204MB
Dimensions:   (ESM: 10, lat: 9, lon: 10, time: 11315)
Coordinates:
  * ESM       (ESM) object 80B 'ACCESS-CM2' 'BCC-CSM2-MR' ... 'UKESM1-0-LL'
  * lat       (lat) int64 72B 47 48 49 50 51 52 53 54 55
  * lon       (lon) int64 80B 6 7 8 9 10 11 12 13 14 15
  * time      (time) datetime64[ns] 91kB 1970-01-01 1970-01-02 ... 2000-12-31
    crs       int64 8B 4326
    gridtype  <U6 24B 'lonlat'
    run       (ESM) object 80B 'r1i1p1f1' 'r2i1p1f1' ... 'r1i1p1f1' 'r1i1p1f2'
    ESM_run   (ESM) object 80B 'ACCESS-CM2_r1i1p1f1' ... 'UKESM1-0-LL_r1i1p1f2'
Data variables:
    tas       (ESM, time, lat, lon) float32 41MB dask.array<chunksize=(1, 11315, 9, 10), meta=np.ndarray>
    rsds      (ESM, time, lat, lon) float32 41MB dask.array<chunksize=(1, 11315, 9, 10), meta=np.ndarray>
    sfcWind   (ESM, time, lat, lon) float64 81MB dask.array<chunksize=(1, 11315, 9, 10), meta=np.ndarray>
    tasmax    (ESM, time, lat, lon) float32 41MB dask.array<chunksize=(1, 11315, 9, 10), meta=np.ndarray>
Attributes:
    regrid_method:  bilinear

In [7]:
# Read the dataframe from the csv file
# used_runs_old = pd.read_csv('/home/onennecke/CMIP_models/CMIP6_runs.csv')
# used_runs_old
used_runs_CMIP = pd.read_csv('/home/onennecke/CMIP_models/CMIP6_runs_future.csv')

used_runs_CESM = pd.read_csv('/home/onennecke/CMIP_models/CESM2_LE_runs.csv')

used_runs = pd.concat([used_runs_CMIP, used_runs_CESM], ignore_index=True)

# Change the ref column to 1 for the first instance of each model
# used_runs['Ref'] = used_runs.groupby(['ESM', 'Institution']).cumcount().apply(lambda x: 1 if x == 0 else 0)

used_runs['ESM_run'] = used_runs['ESM'] + '_' + used_runs['run']

used_runs

,ESM,Institution,run,ESM_run
0,ACCESS-CM2,CSIRO-ARCCSS,r4i1p1f1,ACCESS-CM2_r4i1p1f1
1,ACCESS-CM2,CSIRO-ARCCSS,r5i1p1f1,ACCESS-CM2_r5i1p1f1
2,ACCESS-CM2,CSIRO-ARCCSS,r1i1p1f1,ACCESS-CM2_r1i1p1f1
3,BCC-CSM2-MR,BCC,r1i1p1f1,BCC-CSM2-MR_r1i1p1f1
4,CESM2,NCAR,r4i1p1f1,CESM2_r4i1p1f1
...,...,...,...,...
187,CESM2,NCAR,LE2-1301_016,CESM2_LE2-1301_016
188,CESM2,NCAR,LE2-1301_017,CESM2_LE2-1301_017
189,CESM2,NCAR,LE2-1301_018,CESM2_LE2-1301_018
190,CESM2,NCAR,LE2-1301_019,CESM2_LE2-1301_019


In [ ]:
variables = ['sfcWind', 'rsds', 'tas', 'tasmax', 'psl'] # List of variables
output_path = '/climca/people/onennecke/debiased_data_LS_cc'
base_path = '/climca/people/onennecke/not_debiased_data'

# def one_run(i):
for i in range(len(used_runs)):
    run_time = time.time()
    ESM = used_runs['ESM'][i]
    Inst = used_runs['Institution'][i]
    run = used_runs['run'][i]
    ESM_run = used_runs['ESM_run'][i]
    # variables_available = used_runs['variables'][i].split(', ')
    print(f'Processing Run Nr. {i+1}, {ESM}, {Inst}, {run}, \n')

    for var in variables:
        output_file = os.path.join(output_path, f"{ESM_run}_{var}.nc")
        if os.path.exists(output_file):
            print(f"File {output_file} already exists. Skipping...")
            continue
        # Process the variable
        file = glob.glob(os.path.join(base_path, f"{ESM_run}_{var}.nc"))
        CMIP6_cc = xr.open_dataset(file[0])[var].load()

        if var == 'psl':
            ERA5_hist = psl_ds_ERA5_1970_2000[var].sel(ESM='ERA5').load()
            CMIP6_hist = psl_ds_CMIP6_1970_2000[var].sel(ESM=ESM).load()
        else:
            ERA5_hist = combined_ds_ERA5_1970_2000[var].sel(ESM='ERA5').load()
            CMIP6_hist = combined_ds_CMIP6_1970_2000[var].sel(ESM=ESM).load()

        # Check shapes
        # print(f"ERA5 shape: {ERA5_hist.shape}, CMIP6 shape: {CMIP6_hist.shape}")
        # print(f"CMIP6 cc shape: {CMIP6_cc.shape}")

        # Apply Linear Scaling bias correction
        debiaser_LS = LinearScaling.from_variable(variable = var)

        obs_hist = ERA5_hist.values
        cm_hist = CMIP6_hist.values
        cm_fut = CMIP6_cc.values

        val_debiased_LS = debiaser_LS.apply(obs_hist, cm_hist, cm_fut)

        # Save the debiased data to a new NetCDF file
        debiased_ds = xr.Dataset({var: (('time', 'lat', 'lon'), val_debiased_LS)},
                                  coords={'time': CMIP6_cc['time'],
                                          'lat': CMIP6_cc['lat'],
                                          'lon': CMIP6_cc['lon'],
                                          'ESM_run': ESM_run})
        debiased_ds.to_netcdf(output_file)

        # Close the datasets to free up memory
        ERA5_hist.close()
        CMIP6_hist.close()
        CMIP6_cc.close()

        # break
    # break



/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for variable rsds in debiaser LinearScaling are currently still experimental and may not have been evaluated in the peer-reviewed literature. Please review the results with care!
  return super()._from_variable(


Processing Run Nr. 1, ACCESS-CM2, CSIRO-ARCCSS, r4i1p1f1, 

File /climca/people/onennecke/debiased_data_LS_cc/ACCESS-CM2_r4i1p1f1_sfcWind.nc already exists. Skipping...
ERA5 shape: (11315, 9, 10), CMIP6 shape: (11315, 9, 10)
CMIP6 cc shape: (3650, 9, 10)


100%|██████████| 90/90 [00:00<00:00, 8002.36it/s]


ERA5 shape: (11315, 9, 10), CMIP6 shape: (11315, 9, 10)
CMIP6 cc shape: (3650, 9, 10)


100%|██████████| 90/90 [00:00<00:00, 11168.93it/s]


ERA5 shape: (11315, 9, 10), CMIP6 shape: (11315, 9, 10)
CMIP6 cc shape: (3650, 9, 10)


... (1,909 outputs left out)

ERA5 shape: (11315, 9, 10), CMIP6 shape: (11315, 9, 10)
CMIP6 cc shape: (3650, 9, 10)


100%|██████████| 90/90 [00:00<00:00, 9159.20it/s]
/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for variable psl in debiaser LinearScaling are currently still experimental and may not have been evaluated in the peer-reviewed literature. Please review the results with care!
  return super()._from_variable(


ERA5 shape: (11315, 40, 50), CMIP6 shape: (11315, 40, 50)
CMIP6 cc shape: (3650, 40, 50)


100%|██████████| 2000/2000 [00:00<00:00, 16245.28it/s]


### Same for +2K GWL

In [30]:
variables = ['sfcWind', 'rsds', 'tas', 'tasmax', 'psl'] # List of variables
output_path = '/climca/people/onennecke/debiased_data_LS_fut'
base_path = '/climca/people/onennecke/not_debiased_data_future_2005_2024_ref'

# def one_run(i):
for i in range(len(used_runs)):
    run_time = time.time()
    ESM = used_runs['ESM'][i]
    Inst = used_runs['Institution'][i]
    run = used_runs['run'][i]
    ESM_run = used_runs['ESM_run'][i]
    # variables_available = used_runs['variables'][i].split(', ')
    print(f'Processing Run Nr. {i+1}, {ESM}, {Inst}, {run}, \n')

    for var in variables:
        output_file = os.path.join(output_path, f"{ESM_run}_{var}.nc")
        if os.path.exists(output_file):
            print(f"File {output_file} already exists. Skipping...")
            continue
        # Process the variable
        file = glob.glob(os.path.join(base_path, f"{ESM_run}_{var}.nc"))
        CMIP6_cc = xr.open_dataset(file[0])[var].load()

        if var == 'psl':
            ERA5_hist = psl_ds_ERA5_1970_2000[var].sel(ESM='ERA5').load()
            CMIP6_hist = psl_ds_CMIP6_1970_2000[var].sel(ESM=ESM).load()
        else:
            ERA5_hist = combined_ds_ERA5_1970_2000[var].sel(ESM='ERA5').load()
            CMIP6_hist = combined_ds_CMIP6_1970_2000[var].sel(ESM=ESM).load()

        # Check shapes
        # print(f"ERA5 shape: {ERA5_hist.shape}, CMIP6 shape: {CMIP6_hist.shape}")
        # print(f"CMIP6 cc shape: {CMIP6_cc.shape}")

        # Apply Linear Scaling bias correction
        debiaser_LS = LinearScaling.from_variable(variable = var)

        obs_hist = ERA5_hist.values
        cm_hist = CMIP6_hist.values
        cm_fut = CMIP6_cc.values

        val_debiased_LS = debiaser_LS.apply(obs_hist, cm_hist, cm_fut)

        # Save the debiased data to a new NetCDF file
        debiased_ds = xr.Dataset({var: (('time', 'lat', 'lon'), val_debiased_LS)},
                                  coords={'time': CMIP6_cc['time'],
                                          'lat': CMIP6_cc['lat'],
                                          'lon': CMIP6_cc['lon'],
                                          'ESM_run': ESM_run})
        debiased_ds.to_netcdf(output_file)

        # Close the datasets to free up memory
        ERA5_hist.close()
        CMIP6_hist.close()
        CMIP6_cc.close()

    #     break
    # break



/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for variable sfcWind in debiaser LinearScaling are currently still experimental and may not have been evaluated in the peer-reviewed literature. Please review the results with care!
  return super()._from_variable(


Processing Run Nr. 1, ACCESS-CM2, CSIRO-ARCCSS, r4i1p1f1, 



100%|██████████| 90/90 [00:00<00:00, 11744.36it/s]


/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for variable rsds in debiaser LinearScaling are currently still experimental and may not have been evaluated in the peer-reviewed literature. Please review the results with care!
  return super()._from_variable(
100%|██████████| 90/90 [00:00<00:00, 17398.94it/s]
/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for variable psl in debiaser LinearScaling are currently still experimental and may not have been evaluated in the peer-reviewed literature. Please review the results with care!
  return super()._from_variable(
100%|██████████| 2000/2000 [00:00<00:00, 16080.51it/s]
/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for variable sfcWind in debiaser LinearScaling are cu

Processing Run Nr. 2, ACCESS-CM2, CSIRO-ARCCSS, r5i1p1f1, 



100%|██████████| 90/90 [00:00<00:00, 11147.16it/s]
/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for variable rsds in debiaser LinearScaling are currently still experimental and may not have been evaluated in the peer-reviewed literature. Please review the results with care!
  return super()._from_variable(
100%|██████████| 90/90 [00:00<00:00, 8604.68it/s]
/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for variable psl in debiaser LinearScaling are currently still experimental and may not have been evaluated in the peer-reviewed literature. Please review the results with care!
  return super()._from_variable(
100%|██████████| 2000/2000 [00:00<00:00, 16395.43it/s]
/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for 

... (376 outputs left out)

Processing Run Nr. 191, CESM2, NCAR, LE2-1301_019, 



100%|██████████| 90/90 [00:00<00:00, 12239.00it/s]
/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for variable rsds in debiaser LinearScaling are currently still experimental and may not have been evaluated in the peer-reviewed literature. Please review the results with care!
  return super()._from_variable(
100%|██████████| 90/90 [00:00<00:00, 15284.13it/s]
/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for variable psl in debiaser LinearScaling are currently still experimental and may not have been evaluated in the peer-reviewed literature. Please review the results with care!
  return super()._from_variable(
100%|██████████| 2000/2000 [00:00<00:00, 14804.20it/s]
/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for

Processing Run Nr. 192, CESM2, NCAR, LE2-1301_020, 



100%|██████████| 90/90 [00:00<00:00, 15201.65it/s]
/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for variable rsds in debiaser LinearScaling are currently still experimental and may not have been evaluated in the peer-reviewed literature. Please review the results with care!
  return super()._from_variable(
100%|██████████| 90/90 [00:00<00:00, 12900.70it/s]
/home/onennecke/.conda/envs/env_ma_on/lib/python3.12/site-packages/ibicus/debias/_linear_scaling.py:112: UserWarning: The default settings for variable psl in debiaser LinearScaling are currently still experimental and may not have been evaluated in the peer-reviewed literature. Please review the results with care!
  return super()._from_variable(
100%|██████████| 2000/2000 [00:00<00:00, 15495.66it/s]
